In [2]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd

from src.data_loader import load_data
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler

In [3]:
df = load_data("../data/raw/dataset.csv") 

In [4]:
# antes de fazer a limpeza, vamos criar uma cópia do dataframe original para não perder os dados originais
df_limpo = df.copy()

In [5]:
# Identificando as colunas de texto
colunas_texto = df_limpo.select_dtypes(include="str").columns
colunas_texto

# Convertendo para string e removendo espaços extras
for col in colunas_texto:
    df_limpo[col] = df_limpo[col].astype(str).str.strip()

print("Espaços removidos das colunas de texto.")

Espaços removidos das colunas de texto.


In [6]:
colunas_numericas = [
    "Age", "Height_cm", "Weight_kg", "Sleep_Hours",
    "Exercise_Hours_Per_Week", "HbA1c",
    "Daily_Walking_Minutes", "Blood_Glucose",
    "Total_Cholesterol", "HDL", "LDL", "Triglycerides"
]

resumo = df_limpo[colunas_numericas].agg(
    ["mean", "median", "std", "min", "max"]
).T

resumo["assimetria"] = df_limpo[colunas_numericas].skew()
print(resumo.round(2))

                           mean  median    std    min    max  assimetria
Age                       53.87    54.0  21.13   18.0   90.0        0.01
Height_cm                170.02   170.0  14.46  145.0  195.0        0.00
Weight_kg                 87.46    87.3  24.50   45.0  130.0        0.00
Sleep_Hours                6.50     6.5   2.03    3.0   10.0       -0.01
Exercise_Hours_Per_Week    5.02     5.0   2.88    0.0   10.0       -0.01
HbA1c                      8.49     8.5   2.31    4.5   12.5        0.00
Daily_Walking_Minutes     89.93    89.8  52.03    0.0  180.0        0.00
Blood_Glucose            159.91   159.9  52.04   70.0  250.0        0.01
Total_Cholesterol        220.17   220.4  57.79  120.0  320.0       -0.01
HDL                       57.37    57.3  18.72   25.0   90.0        0.00
LDL                      135.01   135.1  49.14   50.0  220.0       -0.00
Triglycerides            229.54   229.5  98.23   60.0  400.0       -0.00


Variáveis numéricas - As análises exploratórias mostraram que as médias e medianas
são próximas e que as distribuições apresentam assimetria
próxima de zero. Portanto, tanto a média quanto a mediana
seriam adequadas para a imputação.
Optou-se pela mediana por ser uma medida robusta de tendência
central e por permitir um procedimento consistente de tratamento.

In [7]:
# Variáveis numéricas que apresentam valores ausentes
colunas_numericas = [
    "Age", "Height_cm", "Weight_kg", "Sleep_Hours",
    "Exercise_Hours_Per_Week", "HbA1c",
    "Daily_Walking_Minutes", "Blood_Glucose",
    "Total_Cholesterol", "HDL", "LDL", "Triglycerides"
]

# Imputação dos valores ausentes pela mediana
for coluna in colunas_numericas:

    mediana = df_limpo[coluna].median()

    # Substitui os valores ausentes pela mediana da variável,
    # preservando os valores que já estavam preenchidos.
    df_limpo[coluna] = df_limpo[coluna].fillna(mediana)


Variáveis categóricas - Substitui os valores ausentes por uma categoria explícita.
A escolha de "Não_informado" preserva a informação de que
o valor original não estava disponível, evitando atribuir
uma categoria arbitrária por meio da moda.

In [8]:
# Variáveis categóricas que apresentam valores ausentes
colunas_categoricas = [
    "Physical_Activity_Level",
    "Medication_Adherence"
]

for coluna in colunas_categoricas:
    df_limpo[coluna] = df_limpo[coluna].fillna("Não_informado")

In [9]:
# Conta os valores ausentes de todas as colunas
nulos_por_coluna = df_limpo.isnull().sum()

# Exibe somente as colunas que ainda possuem valores ausentes
nulos_restantes = nulos_por_coluna[nulos_por_coluna > 0]

if nulos_restantes.empty:
    print("Nenhum valor nulo restante no DataFrame.")
else:
    print("Ainda existem valores nulos:")
    print(nulos_restantes)


Nenhum valor nulo restante no DataFrame.


In [10]:
# BMI
outliers_bmi = df_limpo[
    (df_limpo["BMI"] < 0.05) |
    (df_limpo["BMI"] > 60.45)
]

# Diabetes_Risk_Score
outliers_risk = df_limpo[
    (df_limpo["Diabetes_Risk_Score"] < 31) |
    (df_limpo["Diabetes_Risk_Score"] > 119)
]

print("Outliers de BMI:", len(outliers_bmi))
print("Outliers de Diabetes_Risk_Score:", len(outliers_risk))

Outliers de BMI: 28
Outliers de Diabetes_Risk_Score: 237


In [11]:
print(outliers_bmi["BMI"].sort_values().to_list())

print(
    outliers_risk["Diabetes_Risk_Score"]
    .sort_values()
    .to_list()
)

[60.5, 60.5, 60.5, 60.5, 60.5, 60.5, 60.6, 60.6, 60.6, 60.6, 60.6, 60.7, 60.7, 60.7, 60.8, 60.8, 60.8, 60.8, 61.0, 61.0, 61.0, 61.1, 61.1, 61.2, 61.3, 61.4, 61.6, 61.7]
[13, 16, 16, 16, 17, 17, 18, 18, 18, 18, 18, 18, 19, 19, 19, 19, 19, 19, 20, 20, 20, 20, 21, 21, 21, 21, 21, 21, 21, 21, 22, 22, 22, 22, 22, 22, 22, 22, 22, 23, 23, 23, 23, 23, 23, 23, 23, 23, 23, 23, 23, 23, 23, 23, 23, 23, 23, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 25, 25, 25, 25, 25, 25, 25, 25, 25, 25, 25, 25, 25, 25, 25, 25, 25, 25, 25, 25, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 30, 30, 30

In [12]:
colunas_one_hot = ["Gender", "Work_Type"]

encoder = OneHotEncoder(
    sparse_output=False,
    dtype=int
)

dados_codificados = encoder.fit_transform(
    df_limpo[colunas_one_hot]
)

nomes_colunas = encoder.get_feature_names_out(colunas_one_hot)

df_encoded = pd.DataFrame(
    dados_codificados,
    columns=nomes_colunas,
    index=df_limpo.index
)

df_limpo = df_limpo.join(df_encoded)

In [13]:
# Variáveis binárias com categorias Yes e No
variaveis_binarias = [
    "Doctor_Consultation_Needed",
    "PCOS",
    "Fatty_Liver",
    "Heart_Disease",
    "Hypertension",
    "Family_History_Diabetes"
]

# Mapeamento binário: No = 0, Yes = 1
mapeamento_binario = {
    "No": 0,
    "Yes": 1
}

# Criar novas colunas, preservando as originais
for coluna in variaveis_binarias:
    df_limpo[f"{coluna}_Encoded"] = (
        df_limpo[coluna].map(mapeamento_binario)
    )

# Codificar Residence_Type em uma nova coluna
df_limpo["Residence_Type_Encoded"] = (
    df_limpo["Residence_Type"].map({
        "Rural": 0,
        "Urban": 1
    })
)

In [14]:
colunas_codificadas = [
    f"{coluna}_Encoded"
    for coluna in variaveis_binarias
] + ["Residence_Type_Encoded"]

print(df_limpo[colunas_codificadas].head())

print("\nValores ausentes nas novas colunas:")
print(df_limpo[colunas_codificadas].isna().sum())

   Doctor_Consultation_Needed_Encoded  PCOS_Encoded  Fatty_Liver_Encoded  \
0                                   1             0                    1   
1                                   1             1                    1   
2                                   1             0                    1   
3                                   1             0                    0   
4                                   1             1                    1   

   Heart_Disease_Encoded  Hypertension_Encoded  \
0                      1                     0   
1                      1                     0   
2                      0                     0   
3                      0                     1   
4                      1                     1   

   Family_History_Diabetes_Encoded  Residence_Type_Encoded  
0                                1                       0  
1                                1                       0  
2                                1                       1 

In [15]:
variaveis_binarias = [
    "Doctor_Consultation_Needed",
    "PCOS",
    "Fatty_Liver",
    "Heart_Disease",
    "Hypertension",
    "Family_History_Diabetes"
]

for coluna in variaveis_binarias:
    print(f"{coluna}: {df_limpo[coluna].unique()}")

Doctor_Consultation_Needed: <StringArray>
['Yes', 'No']
Length: 2, dtype: str
PCOS: <StringArray>
['No', 'Yes']
Length: 2, dtype: str
Fatty_Liver: <StringArray>
['Yes', 'No']
Length: 2, dtype: str
Heart_Disease: <StringArray>
['Yes', 'No']
Length: 2, dtype: str
Hypertension: <StringArray>
['No', 'Yes']
Length: 2, dtype: str
Family_History_Diabetes: <StringArray>
['Yes', 'No']
Length: 2, dtype: str


In [16]:
mapeamentos_ordinais = {
    "Physical_Activity_Level": ["Low", "Moderate", "High", "Não_informado"],
    "Diet_Quality": ["Poor", "Average", "Healthy"],
    "Sugar_Intake_Level": ["Low", "Moderate", "High"],
    "Stress_Level": ["Low", "Moderate", "High"],
    "Alcohol_Consumption": ["Never", "Occasionally", "Frequently"],
    "Medication_Adherence": ["Poor", "Average", "Good", "Não_informado"],
    "Diabetes_Risk": ["Low", "Moderate", "High"],
    "BMI_CATEGORY": ["Underweight", "Normal", "Overweight", "Obese"]
}

colunas = list(mapeamentos_ordinais.keys())

encoder = OrdinalEncoder(
    categories=list(mapeamentos_ordinais.values()),
    dtype=int
)

colunas_encoded = [f"{coluna}_Encoded" for coluna in colunas]

df_limpo[colunas_encoded] = encoder.fit_transform(
    df_limpo[colunas]
)

# Codificar "Não_informado" como -1 apenas nas duas colunas
for coluna in ["Physical_Activity_Level", "Medication_Adherence"]:
    df_limpo.loc[
        df_limpo[coluna] == "Não_informado",
        f"{coluna}_Encoded"
    ] = -1

In [17]:
colunas_numericas = [
    "Age",
    "Height_cm",
    "Weight_kg",
    "BMI",
    "Waist_Circumference_cm",
    "Blood_Glucose",
    "HbA1c",
    "Fasting_Blood_Sugar",
    "Insulin_Level",
    "Blood_Pressure_Systolic",
    "Blood_Pressure_Diastolic",
    "Total_Cholesterol",
    "HDL",
    "LDL",
    "Triglycerides",
    "Heart_Rate",
    "Exercise_Hours_Per_Week",
    "Daily_Walking_Minutes",
    "Sleep_Hours",
    "Daily_Water_Intake_L",
]

scaler = StandardScaler()

colunas_scaled = [
    f"{coluna}_Scaled"
    for coluna in colunas_numericas
]

df_limpo[colunas_scaled] = scaler.fit_transform(
    df_limpo[colunas_numericas]
)

In [19]:
df_limpo.to_csv(
    "../data/processed/dataset_clean.csv",
    index=False
)